# Spatial Omics Fusion — Full Pipeline Walkthrough

This notebook walks through the entire pipeline of our spatial transcriptomics domain detection system:

**Part 1 — Data Pipeline**: How we go from raw gene expression counts to model-ready tensors  
**Part 2 — Model Architecture**: What each component does and what its outputs look like  
**Part 3 — Results & Interpretability**: Attention weights, gate values, and ablation comparisons

**Dataset**: Human DLPFC (Dorsolateral Prefrontal Cortex) — 12 tissue slices, ~3,500-4,800 spots each, 7 cortical layers  
**Goal**: Predict which brain layer each spot belongs to, using gene expression + spatial context

In [ ]:
import os, sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from scipy.sparse import issparse
from sklearn.manifold import TSNE
import scanpy as sc
import squidpy as sq
import warnings
warnings.filterwarnings("ignore")

# Project imports
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.data.dataset import load_dlpfc_data
from src.models.model import SpatialOmicsFusion

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

# Constants
SAMPLE_ID = "151673"
DOMAIN_COLORS = ["#E41A1C", "#FF7F00", "#FFD700", "#4DAF4A", "#377EB8", "#984EA3", "#A65628"]
DOMAIN_NAMES = ["Layer1", "Layer2", "Layer3", "Layer4", "Layer5", "Layer6", "WM"]

print(f"Working with slice {SAMPLE_ID}")

---
# Part 1: Data Pipeline

Our input is a spatial transcriptomics tissue slice — a grid of ~3,600 spots on brain tissue, where each spot measures the expression of ~33,000 genes. We need to transform this into a graph-structured dataset suitable for a GNN.

**Pipeline**: Raw counts → Filter → Normalize → Log → HVG selection → Scale → Build KNN graph

## 1.1 Load Raw Data

Each `.h5ad` file contains the raw count matrix (spots x genes) plus spatial coordinates and ground truth layer annotations from Maynard et al., *Nature Neuroscience* 2021.

In [ ]:
# Load the raw h5ad file
adata_raw = sc.read_h5ad(f"../data/raw/{SAMPLE_ID}.h5ad")
print(f"Raw shape: {adata_raw.shape[0]} spots x {adata_raw.shape[1]} genes")
print(f"Spatial coords: {adata_raw.obsm['spatial'].shape}")
print(f"Ground truth labels: {adata_raw.obs['sce.layer_guess'].value_counts().to_dict()}")

# Drop unlabeled spots
adata = adata_raw[~adata_raw.obs["sce.layer_guess"].isna()].copy()
sc.pp.filter_genes(adata, min_cells=3)
print(f"\nAfter filtering: {adata.shape[0]} spots x {adata.shape[1]} genes")

## 1.2 Preprocessing: Normalize → Log → HVG → Scale

Three transformations, each solving a specific problem:

1. **Normalize** (target_sum=10k): Different spots capture different amounts of mRNA. Normalizing makes expression levels comparable across spots — like TF normalization in NLP.
2. **Log1p**: Gene expression follows a power-law distribution (most genes low, a few very high). `log(1+x)` compresses this into something more Gaussian, which neural networks handle better.
3. **HVG + Scale**: Most of 33K genes are uninformative (expressed similarly everywhere). We keep the top 3,000 most variable genes, then z-score them to zero mean, unit variance.

In [ ]:
# Snapshot raw counts before any processing
raw_expr = adata.X.toarray() if issparse(adata.X) else adata.X.copy()

# Step 1 & 2: Normalize + Log
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
norm_expr = adata.X.toarray() if issparse(adata.X) else adata.X.copy()

# Step 3: HVG selection + scaling
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
hvg_mask = adata.var.highly_variable.values
adata = adata[:, hvg_mask].copy()
sc.pp.scale(adata, max_value=10)
final_expr = adata.X.toarray() if issparse(adata.X) else adata.X.copy()

print(f"Raw:   {raw_expr.shape} — values in [{raw_expr.min():.0f}, {raw_expr.max():.0f}]")
print(f"Norm:  {norm_expr.shape} — values in [{norm_expr.min():.2f}, {norm_expr.max():.2f}]")
print(f"Final: {final_expr.shape} — values in [{final_expr.min():.2f}, {final_expr.max():.2f}]")

In [ ]:
# Visualize: expression heatmaps at each stage
np.random.seed(42)
spot_idx = np.sort(np.random.choice(raw_expr.shape[0], 100, replace=False))
hvg_indices = np.where(hvg_mask)[0][:50]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = axes[0].imshow(raw_expr[np.ix_(spot_idx, hvg_indices)],
                      aspect="auto", cmap="viridis", interpolation="nearest")
axes[0].set_title("Step 1: Raw Counts", fontsize=13, fontweight="bold")
axes[0].set_xlabel("50 genes (HVGs)")
axes[0].set_ylabel("100 spots (sampled)")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(norm_expr[np.ix_(spot_idx, hvg_indices)],
                      aspect="auto", cmap="viridis", interpolation="nearest")
axes[1].set_title("Step 2: Normalize + Log1p", fontsize=13, fontweight="bold")
axes[1].set_xlabel("50 genes (HVGs)")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(final_expr[np.ix_(spot_idx, np.arange(50))],
                      aspect="auto", cmap="RdBu_r", interpolation="nearest", vmin=-3, vmax=3)
axes[2].set_title("Step 3: HVG + Scale (z-scored)", fontsize=13, fontweight="bold")
axes[2].set_xlabel("50 / 3000 HVGs")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle(f"Expression Matrix at Each Preprocessing Step", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize: how the distribution changes at each step
raw_nonzero = raw_expr[raw_expr > 0].flatten()
norm_nonzero = norm_expr[norm_expr > 0].flatten()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(raw_nonzero, bins=100, color="#377EB8", alpha=0.8, edgecolor="none")
axes[0].set_title("Raw Counts (nonzero only)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Expression value")
axes[0].set_ylabel("Frequency")
axes[0].set_yscale("log")

axes[1].hist(norm_nonzero, bins=100, color="#4DAF4A", alpha=0.8, edgecolor="none")
axes[1].set_title("After Normalize + Log1p", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Expression value")

axes[2].hist(final_expr.flatten(), bins=100, color="#984EA3", alpha=0.8, edgecolor="none")
axes[2].set_title("After HVG + Scale (z-scored)", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Expression value")

plt.suptitle("Expression Distribution at Each Step", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 1.3 Spatial Graph Construction

Each spot has (x, y) coordinates on the tissue. We connect each spot to its **k=6 nearest spatial neighbors** using a KNN graph. This graph tells the GNN which spots are physically nearby — enabling it to learn spatial context.

The graph is stored in PyTorch Geometric's `edge_index` format: a `[2, num_edges]` tensor where each column is a (source, target) pair.

In [ ]:
# Load processed data (already preprocessed by src/data/preprocess.py)
data = load_dlpfc_data(SAMPLE_ID, processed_dir="../data/processed", seed=42)
coords = data.pos.numpy()
labels = data.y.numpy()
edge_index = data.edge_index.numpy()
n_classes = data.n_classes

print(f"Expression tensor: {data.x.shape}  (spots x HVGs)")
print(f"Edge index:        {data.edge_index.shape}  (2 x edges)")
print(f"Labels:            {data.y.shape}  ({n_classes} classes)")
print(f"Coordinates:       {data.pos.shape}")
print(f"Train/Val/Test:    {data.train_mask.sum()}/{data.val_mask.sum()}/{data.test_mask.sum()}")

In [ ]:
# Visualize: tissue spots and KNN graph
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: spots colored by ground truth domain
for i in range(n_classes):
    mask = labels == i
    axes[0].scatter(coords[mask, 0], coords[mask, 1], c=DOMAIN_COLORS[i],
                    label=DOMAIN_NAMES[i], s=10, alpha=0.8, edgecolors="none")
axes[0].set_title("Tissue Spots (ground truth labels)", fontsize=13, fontweight="bold")
axes[0].set_aspect("equal"); axes[0].invert_yaxis()
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9, markerscale=2)

# Right: spots + KNN graph edges
np.random.seed(42)
edge_sample = np.random.choice(edge_index.shape[1], 5000, replace=False)
lines = [[coords[edge_index[0, i]], coords[edge_index[1, i]]] for i in edge_sample]
axes[1].add_collection(LineCollection(lines, colors="#cccccc", linewidths=0.3, alpha=0.5))
for i in range(n_classes):
    mask = labels == i
    axes[1].scatter(coords[mask, 0], coords[mask, 1], c=DOMAIN_COLORS[i],
                    s=10, alpha=0.8, edgecolors="none", zorder=2)
axes[1].set_title(f"Spatial KNN Graph (k=6, {edge_index.shape[1]} edges)", fontsize=13, fontweight="bold")
axes[1].set_aspect("equal"); axes[1].invert_yaxis()
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].autoscale_view()

plt.suptitle(f"Spatial Structure — Slice {SAMPLE_ID}", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Label distribution — note the class imbalance (motivates class-weighted loss)
counts = [np.sum(labels == i) for i in range(n_classes)]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(DOMAIN_NAMES[:n_classes], counts, color=DOMAIN_COLORS[:n_classes],
              edgecolor="white", linewidth=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(count), ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title(f"Label Distribution — {sum(counts)} spots total", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of spots")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

---
# Part 2: Model Architecture

Our model has three components:

```
Gene Expression (3000 dims)
        │
   MLP Encoder ──────────────────────── Expression Embedding (128-dim)
        │                                          │
   GAT Encoder (spatial neighbors)                 │
        │                                          │
   Spatial Embedding (128-dim)                     │
        │                                          │
   Cross-Attention Fusion ◄────────────────────────┘
   Q = Expression, K/V = Spatial
        │
   Classification Head → Domain Prediction
```

1. **Expression Encoder (MLP)**: Compresses 3,000 genes → 128-dim embedding. Captures gene-level signal.
2. **Spatial Encoder (GAT)**: Takes expression embeddings + KNN graph, aggregates neighbor information via attention. Each spot learns "what are my neighbors like?"
3. **Cross-Attention Fusion**: Expression embedding queries its spatial neighbors — selectively attends to the most relevant ones. This is where the two modalities merge.

## 2.1 Load Trained Model & Extract Intermediate Embeddings

We load the best-performing model (MLP + GAT + Cross-Attention) and run it step by step, saving the embedding at each stage to visualize how representations evolve.

In [ ]:
# Helper: load a trained model checkpoint
import yaml
with open("../configs/default.yaml") as f:
    config = yaml.safe_load(f)

def load_model(mode, fusion_type):
    """Load a trained model from results/."""
    model = SpatialOmicsFusion(
        n_genes=data.x.shape[1], n_classes=data.n_classes,
        embed_dim=config["model"]["embed_dim"],
        hidden_dim=config["model"]["expression_encoder"]["hidden_dim"],
        expr_layers=config["model"]["expression_encoder"]["n_layers"],
        gat_heads=config["model"]["spatial_encoder"]["n_heads"],
        gat_layers=config["model"]["spatial_encoder"]["n_layers"],
        fusion_type=fusion_type,
        fusion_heads=config["model"]["fusion"]["n_heads"],
        fusion_layers=config["model"]["fusion"]["n_layers"],
        dropout=0.0, mode=mode,
    )
    state = torch.load(f"../results/{mode}_{fusion_type}_{SAMPLE_ID}/model.pt",
                       map_location="cpu", weights_only=True)
    model.load_state_dict(state)
    model.eval()
    return model

# Load the cross-attention model (our best)
model = load_model("full", "cross_attention")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [ ]:
# Run the model step by step, extracting intermediate embeddings
with torch.no_grad():
    # Step 1: MLP encodes gene expression → 128-dim
    expr_embed = model.expr_encoder(data.x)
    print(f"Expression embedding: {expr_embed.shape}")

    # Step 2: GAT aggregates spatial neighbors → 128-dim
    spatial_embed = model.spatial_encoder(expr_embed, data.edge_index)
    print(f"Spatial embedding:    {spatial_embed.shape}")

    # Step 3: Cross-attention fuses expression + spatial → 128-dim
    fused_embed = model.fusion(expr_embed, spatial_embed, data.edge_index)
    print(f"Fused embedding:      {fused_embed.shape}")

    # Step 4: Classifier → logits
    logits = model.classifier(fused_embed)
    preds = logits.argmax(dim=-1).numpy()
    print(f"Predictions:          {logits.shape} → {n_classes} classes")

## 2.2 Embedding Progression: MLP → GAT → Cross-Attention

t-SNE projection of the 128-dim embeddings at each stage. Watch how the cluster separation improves as we add spatial context and cross-attention fusion.

In [ ]:
# t-SNE for each embedding stage
print("Computing t-SNE (this takes ~30s)...")
tsne_expr = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(expr_embed.numpy())
tsne_spatial = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(spatial_embed.numpy())
tsne_fused = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(fused_embed.numpy())

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
stages = [
    (tsne_expr, "Expression Embedding\n(MLP output)"),
    (tsne_spatial, "Spatial Embedding\n(GAT output)"),
    (tsne_fused, "Fused Embedding\n(Cross-Attention output)"),
]
for ax, (tsne, title) in zip(axes, stages):
    for i in range(n_classes):
        mask = labels == i
        ax.scatter(tsne[mask, 0], tsne[mask, 1], c=DOMAIN_COLORS[i],
                   label=DOMAIN_NAMES[i], s=3, alpha=0.6, edgecolors="none")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])

handles = [mpatches.Patch(color=DOMAIN_COLORS[i], label=DOMAIN_NAMES[i]) for i in range(n_classes)]
fig.legend(handles=handles, loc="lower center", ncol=n_classes, fontsize=10, frameon=False,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle("Embedding Progression Through Model", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

## 2.3 Spatial Predictions: Ground Truth vs. Each Model

How do predictions look on the actual tissue? We compare the MLP-only baseline (no spatial info), GAT-only, and our full cross-attention model.

In [ ]:
# Load predictions from each ablation model
ablations = [
    ("expr_only", "gated", "MLP Only (no spatial)"),
    ("gat_only", "gated", "GAT Only"),
    ("full", "cross_attention", "MLP+GAT+CrossAttn (ours)"),
]

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# Ground truth
for i in range(n_classes):
    mask = labels == i
    axes[0].scatter(coords[mask, 0], coords[mask, 1], c=DOMAIN_COLORS[i],
                    label=DOMAIN_NAMES[i], s=8, alpha=0.8, edgecolors="none")
axes[0].set_title("Ground Truth", fontsize=13, fontweight="bold")
axes[0].set_aspect("equal"); axes[0].invert_yaxis()
axes[0].set_xticks([]); axes[0].set_yticks([])

# Each model
for j, (mode, ftype, name) in enumerate(ablations):
    m = load_model(mode, ftype)
    with torch.no_grad():
        logits_m, _ = m(data.x, data.edge_index)
    preds_m = logits_m.argmax(dim=-1).numpy()

    # Load ARI from saved metrics
    with open(f"../results/{mode}_{ftype}_{SAMPLE_ID}/metrics.json") as f:
        ari = json.load(f).get("test_ari", 0)

    ax = axes[j + 1]
    for i in range(n_classes):
        mask = preds_m == i
        ax.scatter(coords[mask, 0], coords[mask, 1], c=DOMAIN_COLORS[i],
                   s=8, alpha=0.8, edgecolors="none")
    ax.set_title(f"{name}\nARI={ari:.3f}", fontsize=13, fontweight="bold")
    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])

handles = [mpatches.Patch(color=DOMAIN_COLORS[i], label=DOMAIN_NAMES[i]) for i in range(n_classes)]
fig.legend(handles=handles, loc="lower center", ncol=n_classes, fontsize=10, frameon=False,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle(f"Spatial Domain Predictions — Slice {SAMPLE_ID}", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

---
# Part 3: Interpretability

Beyond accuracy, we want to understand **what the model learned**. We look at:
- **Cross-attention weights**: Which spatial neighbors does each spot attend to?
- **Gated fusion values**: Does the model rely more on gene expression or spatial context?
- **Ablation embedding comparison**: How does each component improve representations?

## 3.1 Cross-Attention Weights at Domain Boundaries

At **domain boundaries** (where a spot's neighbors belong to different layers), the cross-attention mechanism must decide which neighbors to trust. We extract attention weights from the first cross-attention layer and visualize them.

- **Star** = the query spot
- **Circle size** = how much attention weight that neighbor receives  
- **Line thickness** = attention strength

If the model is working well, we'd expect it to attend more strongly to same-domain neighbors.

In [ ]:
# Extract cross-attention weights from the first layer
with torch.no_grad():
    fusion = model.fusion
    N = expr_embed.shape[0]
    neighbor_idx, attn_mask = fusion._build_neighbor_index(data.edge_index, N, expr_embed.device)

    kv = spatial_embed[neighbor_idx]   # (N, max_neighbors, d)
    q = expr_embed.unsqueeze(1)        # (N, 1, d)
    _, attn_weights = fusion.cross_attns[0](q, kv, kv, key_padding_mask=attn_mask,
                                             need_weights=True, average_attn_weights=True)
    attn_weights = attn_weights.squeeze(1).numpy()  # (N, max_neighbors)
    neighbor_idx_np = neighbor_idx.numpy()
    attn_mask_np = attn_mask.numpy()

# Find boundary spots (neighbors span multiple domains)
src, tgt = edge_index[0], edge_index[1]
boundary_spots = []
for spot in range(N):
    neighbors = src[tgt == spot]
    if len(neighbors) > 0 and len(set(labels[neighbors])) > 1:
        boundary_spots.append(spot)

# Pick one example per domain class
example_spots = []
for cls in range(n_classes):
    cls_boundary = [s for s in boundary_spots if labels[s] == cls]
    if cls_boundary:
        example_spots.append(cls_boundary[len(cls_boundary) // 2])
example_spots = example_spots[:6]

print(f"Found {len(boundary_spots)} boundary spots, showing {len(example_spots)} examples")

In [ ]:
# Plot attention weights for each boundary spot
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes_flat = axes.flatten()

for panel_idx, spot in enumerate(example_spots):
    ax = axes_flat[panel_idx]
    nbr_ids = neighbor_idx_np[spot]
    nbr_mask = ~attn_mask_np[spot]
    valid_nbrs = nbr_ids[nbr_mask]
    valid_weights = attn_weights[spot][nbr_mask]

    # Plot local neighborhood context (faded)
    spot_coord = coords[spot]
    dists = np.sqrt(np.sum((coords - spot_coord) ** 2, axis=1))
    radius = np.sort(dists)[min(50, len(dists) - 1)]
    local_mask = dists <= radius
    for i in range(n_classes):
        cls_mask = (labels == i) & local_mask
        if cls_mask.sum() > 0:
            ax.scatter(coords[cls_mask, 0], coords[cls_mask, 1],
                       c=DOMAIN_COLORS[i], s=30, alpha=0.2, edgecolors="none")

    # Draw attention edges
    max_w = valid_weights.max() if len(valid_weights) > 0 else 1
    for nbr, w in zip(valid_nbrs, valid_weights):
        lw = 1 + 4 * (w / max_w)
        alpha = 0.3 + 0.7 * (w / max_w)
        ax.plot([spot_coord[0], coords[nbr, 0]], [spot_coord[1], coords[nbr, 1]],
                color="#333333", linewidth=lw, alpha=alpha, zorder=3)

    # Query spot (star)
    ax.scatter([spot_coord[0]], [spot_coord[1]], c=DOMAIN_COLORS[labels[spot]],
               s=150, edgecolors="black", linewidths=2, zorder=5, marker="*")

    # Neighbor spots (sized by attention weight)
    for nbr, w in zip(valid_nbrs, valid_weights):
        size = 40 + 160 * (w / max_w)
        ax.scatter([coords[nbr, 0]], [coords[nbr, 1]], c=DOMAIN_COLORS[labels[nbr]],
                   s=size, edgecolors="black", linewidths=1, zorder=4)

    ax.set_title(f"Spot {spot} ({DOMAIN_NAMES[labels[spot]]})", fontsize=11, fontweight="bold")
    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])

for i in range(len(example_spots), 6):
    axes_flat[i].set_visible(False)

plt.suptitle("Cross-Attention Weights at Domain Boundaries\n"
             "(star=query, circle size=attention weight, line width=attention strength)",
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

## 3.2 Gated Fusion: Expression vs. Spatial Reliance

The Gated Fusion model learns a per-spot gate: `output = gate * expression + (1-gate) * spatial`. A gate value near 1 means the model trusts gene expression more; near 0 means it trusts spatial context more.

This tells us: **are there regions where gene expression alone is sufficient, and regions where spatial context is critical?**

In [ ]:
# Load the gated fusion model and extract gate values
model_gated = load_model("full", "gated")

with torch.no_grad():
    expr_g = model_gated.expr_encoder(data.x)
    spatial_g = model_gated.spatial_encoder(expr_g, data.edge_index)

    # Extract the learned gate values
    gate_values = model_gated.fusion.gate(
        torch.cat([expr_g, spatial_g], dim=-1)
    ).numpy()  # (N, 128)

# Average across embedding dimensions → one value per spot
mean_gate = gate_values.mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Ground truth
for i in range(n_classes):
    mask = labels == i
    axes[0].scatter(coords[mask, 0], coords[mask, 1], c=DOMAIN_COLORS[i],
                    label=DOMAIN_NAMES[i], s=10, alpha=0.8, edgecolors="none")
axes[0].set_title("Ground Truth", fontsize=13, fontweight="bold")
axes[0].set_aspect("equal"); axes[0].invert_yaxis()
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9, markerscale=2)

# Gate value heatmap on tissue
sc_plot = axes[1].scatter(coords[:, 0], coords[:, 1], c=mean_gate, cmap="RdYlBu_r",
                           s=10, alpha=0.8, edgecolors="none", vmin=0.3, vmax=0.7)
axes[1].set_title("Gate Value\n(1=expression, 0=spatial)", fontsize=13, fontweight="bold")
axes[1].set_aspect("equal"); axes[1].invert_yaxis()
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.colorbar(sc_plot, ax=axes[1], fraction=0.046, pad=0.04)

# Gate distribution per domain
for i in range(n_classes):
    mask = labels == i
    axes[2].hist(mean_gate[mask], bins=30, color=DOMAIN_COLORS[i],
                 alpha=0.5, label=DOMAIN_NAMES[i], density=True)
axes[2].set_title("Gate Distribution by Domain", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Mean gate value")
axes[2].set_ylabel("Density")
axes[2].legend(fontsize=9)
axes[2].spines["top"].set_visible(False); axes[2].spines["right"].set_visible(False)

plt.suptitle("Gated Fusion Analysis", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 3.3 Ablation Study: How Each Component Improves Representations

Final comparison — t-SNE of the output embeddings from each ablation model. This visually demonstrates why each component matters:

| Model | What it shows |
|---|---|
| MLP Only | Gene expression alone gives rough clustering but many overlaps |
| GAT Only | Spatial context alone dramatically improves separation |
| MLP+GAT (Gated) | Combining both via gating further tightens clusters |
| MLP+GAT (Cross-Attn) | Selective neighbor attention gives the cleanest separation |

In [ ]:
# Load all ablation models and extract embeddings
ablation_configs = [
    ("expr_only", "gated", "MLP Only\n(no spatial)"),
    ("gat_only", "gated", "GAT Only\n(no expression branch)"),
    ("full", "gated", "MLP+GAT\n(Gated Fusion)"),
    ("full", "cross_attention", "MLP+GAT\n(Cross-Attention)"),
]

embeddings_dict = {}
for mode, ftype, name in ablation_configs:
    m = load_model(mode, ftype)
    with torch.no_grad():
        _, emb = m(data.x, data.edge_index)
    embeddings_dict[name] = emb.numpy()

# t-SNE for each
n = len(embeddings_dict)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))

print("Computing t-SNE for each ablation model...")
for ax, (name, emb) in zip(axes, embeddings_dict.items()):
    tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(emb)
    for i in range(n_classes):
        mask = labels == i
        ax.scatter(tsne[mask, 0], tsne[mask, 1], c=DOMAIN_COLORS[i],
                   label=DOMAIN_NAMES[i], s=3, alpha=0.6, edgecolors="none")
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])

handles = [mpatches.Patch(color=DOMAIN_COLORS[i], label=DOMAIN_NAMES[i]) for i in range(n_classes)]
fig.legend(handles=handles, loc="lower center", ncol=n_classes, fontsize=10, frameon=False,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle("Ablation Study: Embedding Quality Comparison", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

## 3.4 Benchmark Summary

Full results across all 12 DLPFC slices (mean +/- std):

In [ ]:
# Load benchmark results and display as a summary table
import pandas as pd

with open("../results/benchmark_summary.json") as f:
    benchmark = json.load(f)

rows = []
for key, metrics in benchmark.items():
    if isinstance(metrics, dict) and "ari" in metrics:
        rows.append({
            "Model": key,
            "ARI": f"{metrics['ari']['mean']:.3f} +/- {metrics['ari']['std']:.3f}",
            "NMI": f"{metrics['nmi']['mean']:.3f} +/- {metrics['nmi']['std']:.3f}",
            "Accuracy": f"{metrics['accuracy']['mean']:.3f} +/- {metrics['accuracy']['std']:.3f}",
        })

df = pd.DataFrame(rows)
print("Benchmark Results (12 DLPFC slices)")
print("=" * 70)
print(df.to_string(index=False))

---
## Key Takeaways

1. **Spatial context is the dominant signal**: Adding GAT to MLP improves ARI from 0.37 → 0.92
2. **Cross-attention fusion is the best strategy**: Selectively attending to relevant spatial neighbors outperforms simple concatenation and gated fusion
3. **Foundation model (scGPT) underperforms**: Frozen pretrained embeddings need fine-tuning for spatial domain detection — task-specific learning wins
4. **Lightweight architecture**: Only 1.1M parameters, trains in <30 seconds per slice